# 06 — Transformer on Cycle-History Sequences

Same cycle-history input framing as notebook 05, but using the Transformer
architecture. The Transformer now attends across **10 block-level summaries**
(degradation checkpoints over time) instead of 301 raw sensor timesteps within
one 5-minute RW step.

Compare to the single-step Transformer result: **10.25% MAE (k-fold)**.

In [ ]:
import math
import os
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.dataset_cycle import CycleHistoryDataset
from src.models.transformer import SOHTransformer
from src.utils import set_all_seeds

H5_PATH    = '../data/processed/sequences.h5'
CKPT_DIR   = '../checkpoints'
WINDOW     = 10
N_FEATURES = 15

LR           = 5e-4
WEIGHT_DECAY = 1e-4
BATCH_SIZE   = 256
MAX_EPOCHS   = 100
PATIENCE     = 15
SEED         = 42

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
set_all_seeds(SEED)
print(f'PyTorch {torch.__version__}  |  Device: {DEVICE}')

## Dataset

In [ ]:
train_ds = CycleHistoryDataset(H5_PATH, window=WINDOW, split='train')
test_ds  = CycleHistoryDataset(H5_PATH, window=WINDOW, split='test',
                                norm_mean=train_ds.norm_mean,
                                norm_std=train_ds.norm_std)

print('=== Train ===')
train_ds.describe()
print()
print('=== Test ===')
test_ds.describe()

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f'\nTrain batches/epoch: {len(train_loader)}')

## Model

In [ ]:
model = SOHTransformer(
    input_dim=N_FEATURES,
    d_model=64,
    nhead=4,
    num_encoder_layers=3,
    dim_feedforward=256,
    dropout=0.1,
    max_seq_len=WINDOW,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'SOHTransformer(input={N_FEATURES}, seq_len={WINDOW})  —  {n_params:,} parameters')
print(model)

## Training

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=5, factor=0.5, min_lr=1e-6)
criterion = nn.MSELoss()

os.makedirs(CKPT_DIR, exist_ok=True)
ckpt_path = os.path.join(CKPT_DIR, 'SOHTransformer_cycle_best.pt')

history = {'train_loss': [], 'test_mae': [], 'test_rmse': [], 'lr': []}
best_mae = float('inf')
no_improve = 0

print(f'{"Epoch":>6}  {"Train Loss":>12}  {"Test MAE%":>10}  {"Test RMSE%":>11}  {"LR":>10}  {"Time":>6}')
print('-' * 72)

for epoch in range(1, MAX_EPOCHS + 1):
    t0 = time.time()

    # --- train ---
    model.train()
    running = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(xb).squeeze(1), yb)
        loss.backward()
        optimizer.step()
        running += loss.item()
    train_loss = running / len(train_loader)

    # --- evaluate ---
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            all_preds.append(model(xb.to(DEVICE)).squeeze(1).cpu())
            all_targets.append(yb)
    preds   = torch.cat(all_preds)   * 100.0
    targets = torch.cat(all_targets) * 100.0
    test_mae  = (preds - targets).abs().mean().item()
    test_rmse = math.sqrt(((preds - targets) ** 2).mean().item())

    scheduler.step(test_mae)
    current_lr = optimizer.param_groups[0]['lr']
    elapsed = time.time() - t0

    history['train_loss'].append(train_loss)
    history['test_mae'].append(test_mae)
    history['test_rmse'].append(test_rmse)
    history['lr'].append(current_lr)

    print(f'{epoch:>6}  {train_loss:>12.6f}  {test_mae:>10.4f}  {test_rmse:>11.4f}  '
          f'{current_lr:>10.2e}  {elapsed:>5.1f}s')

    if test_mae < best_mae:
        best_mae = test_mae
        no_improve = 0
        torch.save({'epoch': epoch, 'test_mae': test_mae,
                    'model_state_dict': model.state_dict()}, ckpt_path)
        print(f'         ✓ New best — saved')
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f'\nEarly stopping at epoch {epoch}')
            break

print(f'\nBest test MAE: {best_mae:.4f}%')

## Results

In [ ]:
# Load best checkpoint
ckpt = torch.load(ckpt_path, map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

all_preds, all_targets = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        all_preds.append(model(xb.to(DEVICE)).squeeze(1).cpu())
        all_targets.append(yb)
preds   = torch.cat(all_preds).numpy()   * 100.0
targets = torch.cat(all_targets).numpy() * 100.0

final_mae  = np.abs(preds - targets).mean()
final_rmse = np.sqrt(((preds - targets) ** 2).mean())
print(f'Test MAE:  {final_mae:.4f}%')
print(f'Test RMSE: {final_rmse:.4f}%')
print(f'Best epoch: {ckpt["epoch"]}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pred vs actual
ax = axes[0]
ax.scatter(targets, preds, alpha=0.6, s=20)
lims = [min(targets.min(), preds.min()) - 2, max(targets.max(), preds.max()) + 2]
ax.plot(lims, lims, 'r--', lw=1)
ax.set_xlabel('True SOH (%)')
ax.set_ylabel('Predicted SOH (%)')
ax.set_title(f'Transformer Cycle-History: Pred vs True\nMAE={final_mae:.2f}%  RMSE={final_rmse:.2f}%')
ax.grid(alpha=0.3)

# Training curve
ax = axes[1]
ax.plot(history['test_mae'], label='Test MAE')
ax.axhline(10.25, color='gray',   linestyle='--', label='Single-step Transformer (10.25%)')
ax.axhline(7.46,  color='orange', linestyle='--', label='CNN baseline (7.46%)')
ax.set_xlabel('Epoch')
ax.set_ylabel('Test MAE (SOH%)')
ax.set_title('Transformer Cycle-History: Training Curve')
ax.legend()
ax.grid(alpha=0.3)

os.makedirs('../results/figures', exist_ok=True)
fig.tight_layout()
fig.savefig('../results/figures/transformer_cycle_results.png', dpi=150)
plt.show()

## Comparison Table

In [ ]:
# Fill in LSTM cycle MAE after running notebook 05
lstm_cycle_mae  = None   # replace with actual result from 05_lstm_cycle.ipynb
lstm_cycle_rmse = None

print('\n' + '='*72)
print(f'{"Model":<34}  {"Input Shape":<18}  {"MAE%":>7}  {"RMSE%":>7}')
print('-'*72)
print(f'{"MLP (k-fold)":<34}  {"20 stats/step":<18}  {11.62:>7.2f}  {16.13:>7.2f}')
print(f'{"CNN (k-fold)":<34}  {"(301,3)/step":<18}  {7.46:>7.2f}  {12.64:>7.2f}')
print(f'{"LSTM single-step (k-fold)":<34}  {"(301,3)/step":<18}  {10.88:>7.2f}  {15.61:>7.2f}')
print(f'{"Transformer single-step (k-fold)":<34}  {"(301,3)/step":<18}  {10.25:>7.2f}  {15.23:>7.2f}')
lstm_mae_str  = f'{lstm_cycle_mae:>7.2f}'  if lstm_cycle_mae  else f'{"???":>7}'
lstm_rmse_str = f'{lstm_cycle_rmse:>7.2f}' if lstm_cycle_rmse else f'{"???":>7}'
print(f'{"LSTM cycle-history (nb 05)":<34}  {f"({WINDOW},15)/blocks":<18}  {lstm_mae_str}  {lstm_rmse_str}')
print(f'{"Transformer cycle-history (this nb)":<34}  {f"({WINDOW},15)/blocks":<18}  {final_mae:>7.2f}  {final_rmse:>7.2f}')
print('='*72)
print(f'Note: single-step results are 5-fold k-fold val MAE (22 train batteries).')
print(f'      cycle-history results are test set: RW25-28 (40°C OOD) + RW13-14 (skewed-low OOD).')
print(f'      cycle-history test set size: {len(test_ds)} sequences (interpret with caution).')